# YOLOE playground (stage 1, live)

Run the code cell (**Shift+Enter**). Change any control and the overlay re-renders.
- Prompt **none** = prompt-free: the `-pf` weights of the chosen size detect from a built-in 4585-class vocabulary.
- Changing **prompts / weights** rebuilds the model (~1-2 s, then cached).
- Changing **scene / conf / iou / imgsz / max_det / half** only re-runs predict (~20 ms).
- Defaults come from `config.yaml` (`detect:`). **Save** writes `output/yoloe/<tag>.png` + `<tag>.json`.
- The box under the image is the equivalent CLI command, for reproducing a result.

In [ ]:
import json, os, shlex, sys, time
os.environ.setdefault('YOLO_VERBOSE', 'False')  # else ultralytics logs a banner per predict from widget callbacks
import cv2, numpy as np, ipywidgets as W
from IPython.display import display

REPO = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.path.abspath('.')) == 'notebooks' else os.path.abspath('.')
os.chdir(REPO)  # repo-relative paths below; weights resolve to models/ in vision/detect.py
sys.path.insert(0, REPO)
from vision import load_config
from vision.detect import (build_prompt_free_model, build_text_prompt_model, build_visual_prompt_model, detect,
                           load_prompt)
from vision.viz import COLORS, draw_detection

CFG = load_config()['detect']
REFS = 'data/multi_object_scene/refs'
OUT = 'output/yoloe'
SCENES = {'multi-object (BOP, 5 objects)': 'data/multi_object_scene/scene_rgb.png'}
REF_NAMES = sorted(f[:-4] for f in os.listdir(REFS) if f.endswith('.png'))

# ---------------------------------------------------------------- controls
mode    = W.ToggleButtons(options=['text', 'visual', 'none'], description='prompt',
                          tooltips=['text phrases', 'ref image boxes', 'prompt-free: built-in 4585-class vocab (-pf weights)'])
text    = W.Text('mustard bottle, banana', description='text', continuous_update=False,
                 layout=W.Layout(width='600px'), placeholder='comma-separated class names')
refs    = W.SelectMultiple(options=REF_NAMES, value=(REF_NAMES[-1],), description='refs', rows=3)
custom  = W.Text('', description='own ref', continuous_update=False, layout=W.Layout(width='600px'),
                 placeholder='name | /path/to/ref.png | x0,y0,x1,y1   (; for more)')
scene   = W.Dropdown(options=SCENES, description='scene', layout=W.Layout(width='600px'))
scene_p = W.Text('', description='or path', continuous_update=False, layout=W.Layout(width='600px'))
weights = W.Dropdown(options=['yoloe-11s-seg.pt', 'yoloe-11m-seg.pt', 'yoloe-11l-seg.pt'],
                     value=CFG['weights'], description='weights')
conf    = W.FloatSlider(CFG['conf'], min=0.01, max=0.9, step=0.01, description='conf', continuous_update=False)
iou     = W.FloatSlider(CFG['iou'], min=0.1, max=0.95, step=0.05, description='iou (NMS)', continuous_update=False)
imgsz   = W.SelectionSlider(options=[320, 480, 640, 800, 960, 1280], value=CFG['imgsz'], description='imgsz',
                            continuous_update=False)
max_det = W.IntSlider(CFG['max_det'], min=1, max=300, description='max_det', continuous_update=False)
half    = W.Checkbox(CFG['half'], description='half (FP16)')
tag     = W.Text('play', description='tag')
save    = W.Button(description='save', icon='save')
info    = W.HTML()
image   = W.Image(format='png', layout=W.Layout(max_width='100%'))
_models, _last = {}, {}


def prompts():
    if mode.value == 'none':
        return ()
    if mode.value == 'text':
        return tuple(t.strip() for t in text.value.split(',') if t.strip())
    ps = [(n, os.path.join(REFS, n + '.png'), None) for n in refs.value]
    for item in filter(None, (c.strip() for c in custom.value.split(';'))):
        name, path, box = [x.strip() for x in item.split('|')]
        ps.append((name, path, box))
    return tuple(ps)


def get_model(w, ps):
    key = (w, mode.value, ps)
    if key not in _models:
        if mode.value == 'none':  # same size, -pf weights: vocabulary is baked in
            _models[key] = build_prompt_free_model(w.replace('-seg.pt', '-seg-pf.pt'))
        elif mode.value == 'text':
            _models[key] = build_text_prompt_model(w, list(ps), CFG['device'])
        else:
            vp = []
            for name, path, box in ps:
                if box is None:
                    img, bb = load_prompt(path)
                else:
                    img, bb = cv2.imread(path), [float(v) for v in box.split(',')]
                    if img is None:
                        raise FileNotFoundError(path)
                vp.append((name, img, bb))
            _models[key] = build_visual_prompt_model(w, vp, CFG['device'])
    return _models[key]


def cli(ps):
    args = ['.venv/bin/python', 'pipeline/01_detect.py']
    if mode.value == 'none':
        args += ['--prompt-free']
    elif mode.value == 'text':
        for t in ps: args += ['--text', t]
    else:
        for n, p, b in ps:
            args += ['--ref', n] if b is None else ['--prompt', n, p, b]
    args += ['--scene', scene_p.value or scene.value, '--weights', weights.value, '--conf', str(conf.value),
             '--iou', str(iou.value), '--imgsz', str(imgsz.value), '--max-det', str(max_det.value)]
    return ' '.join(shlex.quote(a) for a in args + (['--half'] if half.value else []))


def run(*_):
    try:
        ps = prompts()
        if not ps and mode.value != 'none':
            raise ValueError('no prompts')
        t0 = time.perf_counter()
        model = get_model(weights.value, ps)
        t_build = time.perf_counter() - t0
        img = cv2.imread(scene_p.value or scene.value)
        if img is None:
            raise FileNotFoundError(scene_p.value or scene.value)
        kw = dict(conf=conf.value, iou=iou.value, imgsz=imgsz.value, max_det=max_det.value,
                  half=half.value, device=CFG['device'])
        detect(model, img, **kw)  # warm-up
        times = []
        for _ in range(5):
            t0 = time.perf_counter()
            dets = detect(model, img, **kw)
            times.append((time.perf_counter() - t0) * 1000)
        classes = [p if isinstance(p, str) else p[0] for p in ps] or sorted({d['name'] for d in dets})
        dets.sort(key=lambda d: -d['score'])
        ov = img
        rows = ''
        for i, d in enumerate(dets):
            c = COLORS[classes.index(d['name']) % len(COLORS)]
            if d['mask'] is not None:
                ov = draw_detection(ov, d, c)
            rows += '<tr><td>%d</td><td style="color:rgb(%d,%d,%d)"><b>%s</b></td><td>%.3f</td><td>%s</td><td>%d</td></tr>' % (
                i, c[2], c[1], c[0], d['name'], d['score'], [int(v) for v in d['bbox']],
                int(d['mask'].sum()) if d['mask'] is not None else 0)
        _last.update(img=ov, dets=dets, settings=dict(kw, weights=weights.value, prompts=ps,
                                                          scene=scene_p.value or scene.value, cli=cli(ps)))
        image.value = cv2.imencode('.png', ov)[1].tobytes()
        info.value = ('<b>%d detections</b> &nbsp; model %s %.2f s &nbsp; predict p50 <b>%.1f ms</b>'
                      '<table><tr><th>#</th><th>class</th><th>score</th><th>bbox xyxy</th><th>mask px</th></tr>%s</table>'
                      '<pre>%s</pre>') % (len(dets), 'cached' if t_build < 0.05 else 'built', t_build,
                                          np.median(times), rows or '<tr><td colspan=5>none above conf</td></tr>', cli(ps))
    except Exception as e:
        info.value = '<pre style="color:red">%s: %s</pre>' % (type(e).__name__, e)


def on_save(_):
    if not _last:
        return
    os.makedirs(OUT, exist_ok=True)
    cv2.imwrite(os.path.join(OUT, tag.value + '.png'), _last['img'])
    with open(os.path.join(OUT, tag.value + '.json'), 'w') as f:
        json.dump({'settings': _last['settings'],
                   'detections': [{k: d[k] for k in ('name', 'score', 'bbox')} for d in _last['dets']]},
                  f, indent=2, default=str)
    info.value += '<p>saved -> %s/%s.png + .json</p>' % (OUT, tag.value)


def on_mode(_):
    text.layout.display = '' if mode.value == 'text' else 'none'
    refs.layout.display = custom.layout.display = '' if mode.value == 'visual' else 'none'
    run()


mode.observe(on_mode, 'value')
for w in (text, refs, custom, scene, scene_p, weights, conf, iou, imgsz, max_det, half):
    w.observe(run, 'value')
save.on_click(on_save)
display(W.VBox([W.HBox([mode, weights]), text, refs, custom, scene, scene_p,
                W.HBox([conf, iou]), W.HBox([imgsz, max_det, half]), W.HBox([tag, save]), info, image]))
on_mode(None)